<a href="https://colab.research.google.com/github/Hanzet22/GAN-Furry-Art-Photo/blob/main/GAN_Furry_Art_Photo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ==========================================
# 1. CLONE REPOSITORY RESMI (DAPAT SEMUA SCRIPT)
# ==========================================
import os
print("📥 Cloning Real-ESRGAN Repository...")
if not os.path.exists('Real-ESRGAN'):
    !git clone https://github.com/xinntao/Real-ESRGAN.git
    print("✅ Repository cloned.")

# FIX: Cek apakah kita sudah ada di dalam folder (mencegah nested folder error)
if not os.path.exists('inference_realesrgan.py'):
    %cd Real-ESRGAN
    print("✅ Entered Real-ESRGAN folder.")
else:
    print("✅ Already in Real-ESRGAN folder.")

# ==========================================
# 2. INSTALL DEPENDENCIES (PAKE REQUIREMENTS.TXT RESMI)
# ==========================================
print("🔧 Installing dependencies from requirements.txt...")
!pip install -q basicsr facexlib gfpgan
!pip install -q -r requirements.txt
!python setup.py develop -q
print("✅ Dependencies installed.")

# ==========================================
# 3. DOWNLOAD MODEL ANIME (BUAT FURRY)
# ==========================================
os.makedirs('weights', exist_ok=True)
model_path = 'weights/RealESRGAN_x4plus_anime_6B.pth'

if not os.path.exists(model_path):
    print("⏳ Downloading Model Anime...")
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth -P weights/
    print("✅ Model downloaded.")
else:
    print("✅ Model already exists.")

# ==========================================
# 4. UPLOAD GAMBAR KE FOLDER 'inputs'
# ==========================================
from google.colab import files
import glob

os.makedirs('inputs', exist_ok=True)
os.makedirs('results', exist_ok=True)

print("📂 Please upload your Furry image(s)...")
uploaded = files.upload()

for filename in uploaded.keys():
    new_name = filename.replace(" ", "_")
    !mv "{filename}" "inputs/{new_name}"
    print(f"   ➕ Uploaded: {new_name}")

# ==========================================
# 5. JALANKAN UPSCALE PAKE SCRIPT ASLI (FIXED PARAMS)
# ==========================================
print("\n🚀 Starting Upscale Process (Using Official Script)...")

# FIX: WAJIB pakai --model_name (-n) karena file weights anime_6B itu beda arsitektur sama defaultnya.
# Kalau nggak dikasih -n, script nyoba load weights kecil ke model gede -> ERROR Missing key(s).

!python inference_realesrgan.py \
    --model_path weights/RealESRGAN_x4plus_anime_6B.pth \
    --model_name RealESRGAN_x4plus_anime_6B \
    --input inputs \
    --output results \
    -s 4 \
    --ext png

# ==========================================
# 6. DOWNLOAD HASIL
# ==========================================
print("\n🎉 Process Finished! Preparing download...")

# FIX: Zip langsung dari sini (cwd) ke absolute path /content biar aman
if os.path.exists('results') and os.listdir('results'):
    result_files = glob.glob('results/*.png')
    print(f"Found {len(result_files)} images.")

    !zip -r /content/results.zip results
    files.download('/content/results.zip')
    print("📥 Download started.")
else:
    print("⚠️ No results found. Check error logs above.")